In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

In [2]:
df = pd.read_csv('/home/ali-mirza/AI-ML/data/train.txt',sep=';',header=None,names=['text','emotions'])
df.head()

,text,emotions
0,i didnt feel humiliated,sadness
1,i can go from feeling so hopeless to so damned...,sadness
2,im grabbing a minute to post i feel greedy wrong,anger
3,i am ever feeling nostalgic about the fireplac...,love
4,i am feeling grouchy,anger


In [3]:
df.isnull().sum()

text        0
emotions    0
dtype: int64

In [4]:
df['emotions'].unique()

<ArrowStringArray>
['sadness', 'anger', 'love', 'surprise', 'fear', 'joy']
Length: 6, dtype: str

In [5]:
emotions = df['emotions'].unique()
emotion_num = {}
i = 0
for emo in emotions:
    emotion_num[emo] = i
    i+=1
df['emotions'] = df['emotions'].map(emotion_num)
df['emotions'].unique()

array([0, 1, 2, 3, 4, 5])

In [6]:
df['emotions'].head()

0    0
1    0
2    1
3    2
4    1
Name: emotions, dtype: int64

In [7]:
#Convert all text in LowerCase

df['text'] = df['text'].apply(lambda x : x.lower())
df['text']

0                                  i didnt feel humiliated
1        i can go from feeling so hopeless to so damned...
2         im grabbing a minute to post i feel greedy wrong
3        i am ever feeling nostalgic about the fireplac...
4                                     i am feeling grouchy
                               ...                        
15995    i just had a very brief time in the beanbag an...
15996    i am now turning and i feel pathetic that i am...
15997                       i feel strong and good overall
15998    i feel like this was such a rude comment and i...
15999    i know a lot but i feel so stupid because i ca...
Name: text, Length: 16000, dtype: str

In [8]:
#Now remove Punctuation
import string

def remove_punctuation(txt):
    return txt.translate(str.maketrans("","",string.punctuation))

df['text'] = df['text'].apply(remove_punctuation)

In [9]:
#Now remove Numbers
def remove_digits(txt):
    new = ""
    for i in txt:
        if not i.isdigit():
            new += i
    return new
df['text'] = df['text'].apply(remove_digits)
df['text'].head()

0                              i didnt feel humiliated
1    i can go from feeling so hopeless to so damned...
2     im grabbing a minute to post i feel greedy wrong
3    i am ever feeling nostalgic about the fireplac...
4                                 i am feeling grouchy
Name: text, dtype: str

In [10]:
#Remove links
def remove_links(txt):
    words = txt.split()

    new_words = []

    for word in words:
        if not word.startswith(("http://", "https://", "www.")):
            new_words.append(word)

    return " ".join(new_words)
df['text'] = df['text'].apply(remove_links)
df['text'].head()

0                              i didnt feel humiliated
1    i can go from feeling so hopeless to so damned...
2     im grabbing a minute to post i feel greedy wrong
3    i am ever feeling nostalgic about the fireplac...
4                                 i am feeling grouchy
Name: text, dtype: str

In [11]:
#Remove Emojis

def remove_emojis(txt):
    new = ""
    for i in txt:
        if i.isascii():
            new += i
    return new
df['text'] = df['text'].apply(remove_emojis)
df['text'].head()

0                              i didnt feel humiliated
1    i can go from feeling so hopeless to so damned...
2     im grabbing a minute to post i feel greedy wrong
3    i am ever feeling nostalgic about the fireplac...
4                                 i am feeling grouchy
Name: text, dtype: str

In [12]:
#Now remove Stop Words to reduce noise

import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

nltk.download("punkt_tab")
nltk.download('stopwords')

[nltk_data] Downloading package punkt_tab to /home/ali-
[nltk_data]     mirza/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to /home/ali-
[nltk_data]     mirza/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [13]:
stop_words = set(stopwords.words('english'))
len(stop_words)

198

In [14]:
def remove_stopWords(txt):
    words = word_tokenize(txt)
    cleaned = []
    for word in words:
        if not word in stop_words:
            cleaned.append(word)
    return ' '.join(cleaned)

df['text'] = df['text'].apply(remove_stopWords)
df['text'].head()
            

0                                didnt feel humiliated
1    go feeling hopeless damned hopeful around some...
2            im grabbing minute post feel greedy wrong
3    ever feeling nostalgic fireplace know still pr...
4                                      feeling grouchy
Name: text, dtype: str

In [15]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(df['text'], df['emotions'], test_size=0.20, random_state=42)



In [16]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

bow_vectorizer = CountVectorizer()
tfidf_vectorizer = TfidfVectorizer()

X_train_bow = bow_vectorizer.fit_transform(X_train)
X_test_bow = bow_vectorizer.transform(X_test)

X_train_tfidf = tfidf_vectorizer.fit_transform(X_train)
X_test_tfidf = tfidf_vectorizer.transform(X_test)


In [17]:
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import MultinomialNB
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    classification_report,
    confusion_matrix
)

In [18]:
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "Linear SVM": LinearSVC(),
    "Multinomial NB": MultinomialNB(),
    "Random Forest": RandomForestClassifier(random_state=42),
    "Decision Tree": DecisionTreeClassifier(random_state=42),
    "KNN": KNeighborsClassifier(),
    "AdaBoost": AdaBoostClassifier(random_state=42)
}

In [19]:
bow_result = []

for name,model in models.items():
    model.fit(X_train_bow,y_train)
    bow_pred = model.predict(X_test_bow)
    acc = accuracy_score(y_test,bow_pred)
    f1 = f1_score(y_test,bow_pred,average='macro')
    bow_result.append({
        "name":name,
        "accuracy":acc,
        "f1 score":f1
    })
bow_result

[{'name': 'Logistic Regression',
  'accuracy': 0.88875,
  'f1 score': 0.8561616435862652},
 {'name': 'Linear SVM', 'accuracy': 0.8896875, 'f1 score': 0.8567800825987318},
 {'name': 'Multinomial NB',
  'accuracy': 0.7678125,
  'f1 score': 0.6000032663805691},
 {'name': 'Random Forest',
  'accuracy': 0.8871875,
  'f1 score': 0.8542702462360093},
 {'name': 'Decision Tree',
  'accuracy': 0.87125,
  'f1 score': 0.8349869238775222},
 {'name': 'KNN', 'accuracy': 0.55, 'f1 score': 0.46313596489568454},
 {'name': 'AdaBoost', 'accuracy': 0.319375, 'f1 score': 0.08256921518505643}]

In [20]:
pd.DataFrame(bow_result).sort_values(by='accuracy',ascending=False)

,name,accuracy,f1 score
1,Linear SVM,0.889687,0.856780
0,Logistic Regression,0.888750,0.856162
3,Random Forest,0.887188,0.854270
4,Decision Tree,0.871250,0.834987
2,Multinomial NB,0.767813,0.600003
5,KNN,0.550000,0.463136
6,AdaBoost,0.319375,0.082569


In [21]:
tfidf_result = []

for name,model in models.items():
    model.fit(X_train_tfidf,y_train)
    tfidf_pred = model.predict(X_test_tfidf)
    acc = accuracy_score(y_test,tfidf_pred)
    f1 = f1_score(y_test,tfidf_pred,average='macro')
    tfidf_result.append({
        "name":name,
        "accuracy":acc,
        "f1 score":f1
    })
tfidf_result

[{'name': 'Logistic Regression',
  'accuracy': 0.8615625,
  'f1 score': 0.7998246503671035},
 {'name': 'Linear SVM', 'accuracy': 0.891875, 'f1 score': 0.8576844054518141},
 {'name': 'Multinomial NB',
  'accuracy': 0.6609375,
  'f1 score': 0.40273009324363523},
 {'name': 'Random Forest',
  'accuracy': 0.8809375,
  'f1 score': 0.847219929716723},
 {'name': 'Decision Tree',
  'accuracy': 0.8671875,
  'f1 score': 0.8308126346676818},
 {'name': 'KNN', 'accuracy': 0.780625, 'f1 score': 0.7239290130448138},
 {'name': 'AdaBoost', 'accuracy': 0.31875, 'f1 score': 0.08178917247345555}]

In [22]:
pd.DataFrame(tfidf_result).sort_values(by='accuracy',ascending=False)

,name,accuracy,f1 score
1,Linear SVM,0.891875,0.857684
3,Random Forest,0.880938,0.847220
4,Decision Tree,0.867188,0.830813
0,Logistic Regression,0.861563,0.799825
5,KNN,0.780625,0.723929
2,Multinomial NB,0.660937,0.402730
6,AdaBoost,0.318750,0.081789


In [23]:
from sklearn.model_selection import cross_val_score

top_models = {
    "Linear SVM": LinearSVC(),
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "Random Forest": RandomForestClassifier(random_state=42)
}

In [24]:
cv_result = []

for name,model in top_models.items():
    if name == "Linear SVM":
        X = X_train_tfidf
    else:
        X = X_train_bow
    cv_score = cross_val_score(
        model,
        X,
        y_train,
        cv=5,
        scoring='f1_macro',
        n_jobs=-1
    )
    cv_result.append({
        'name':name,
        'mean f1':cv_score.mean(),
        'std': cv_score.std()
    }
    )
pd.DataFrame(cv_result).sort_values('mean f1',ascending=False)

,name,mean f1,std
0,Linear SVM,0.850024,0.004863
1,Logistic Regression,0.846234,0.004481
2,Random Forest,0.843355,0.006864


In [25]:
param_svm = {
    "C": [0.01, 0.1, 1, 10, 100],
    "loss": ["hinge", "squared_hinge"],
    "class_weight": [None, "balanced"],
    "max_iter": [1000, 2000, 3000, 5000]
}

param_lr = {
    "C": [0.01, 0.1, 1, 10, 100],
    "penalty": ["l2"],
    "solver": ["lbfgs", "liblinear", "saga"],
    "class_weight": [None, "balanced"],
    "max_iter": [500, 1000, 2000]
}

In [26]:
from sklearn.model_selection import RandomizedSearchCV

svm_search = RandomizedSearchCV(
    LinearSVC(),
    param_svm,
    n_iter=20,
    cv=5,
    scoring="f1_macro",
    random_state=42,
    n_jobs=-1
)

lr_search = RandomizedSearchCV(
    LogisticRegression(),
    param_distributions=param_lr,
    n_iter=20,
    cv=5,
    scoring="f1_macro",
    random_state=42,
    n_jobs=-1
)

In [27]:
svm_search.fit(X_train_tfidf, y_train)

,"estimator estimator: estimator objectAn object of that type is instantiated for each grid point.This is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",LinearSVC()
,"param_distributions param_distributions: dict or list of dictsDictionary with parameters names (`str`) as keys and distributionsor lists of parameters to try. Distributions must provide a ``rvs``method for sampling (such as those from scipy.stats.distributions).If a list is given, it is sampled uniformly.If a list of dicts is given, first a dict is sampled uniformly, andthen a parameter is sampled using that dict as above.","{'C': [0.01, 0.1, ...], 'class_weight': [None, 'balanced'], 'loss': ['hinge', 'squared_hinge'], 'max_iter': [1000, 2000, ...]}"
,"n_iter n_iter: int, default=10Number of parameter settings that are sampled. n_iter tradesoff runtime vs quality of the solution.",20
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion <scoring_api_overview>` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.If None, the estimator's score method is used.",'f1_macro'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- an iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide <cross_validation>` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",5
,"random_state random_state: int, RandomState instance or None, default=NonePseudo random number generator state used for random uniform samplingfrom lists of possible values instead of scipy.stats distributions.Pass an int for reproducible output across multiplefunction calls.See :term:`Glossary <random_state>`.",42
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given the ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``RandomizedSearchCV`` instance.Also for multiple metric evalu

In [28]:
lr_search.fit(X_train_bow, y_train)

,"estimator estimator: estimator objectAn object of that type is instantiated for each grid point.This is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",LogisticRegression()
,"param_distributions param_distributions: dict or list of dictsDictionary with parameters names (`str`) as keys and distributionsor lists of parameters to try. Distributions must provide a ``rvs``method for sampling (such as those from scipy.stats.distributions).If a list is given, it is sampled uniformly.If a list of dicts is given, first a dict is sampled uniformly, andthen a parameter is sampled using that dict as above.","{'C': [0.01, 0.1, ...], 'class_weight': [None, 'balanced'], 'max_iter': [500, 1000, ...], 'penalty': ['l2'], ...}"
,"n_iter n_iter: int, default=10Number of parameter settings that are sampled. n_iter tradesoff runtime vs quality of the solution.",20
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion <scoring_api_overview>` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.If None, the estimator's score method is used.",'f1_macro'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- an iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide <cross_validation>` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",5
,"random_state random_state: int, RandomState instance or None, default=NonePseudo random number generator state used for random uniform samplingfrom lists of possible values instead of scipy.stats distributions.Pass an int for reproducible output across multiplefunction calls.See :term:`Glossary <random_state>`.",42
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given the ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``RandomizedSearchCV`` instance.Also for multiple metric evaluatio

In [29]:
print("SVM Best Parameters: ",svm_search.best_params_)
print("SVM Best CV Score: ",svm_search.best_score_)

print("Logistic Regression Best Parameters: ",lr_search.best_params_)
print("Logistic Regression Best CV Score: ",lr_search.best_score_)

SVM Best Parameters:  {'max_iter': 1000, 'loss': 'hinge', 'class_weight': 'balanced', 'C': 1}
SVM Best CV Score:  0.8560062572816205
Logistic Regression Best Parameters:  {'solver': 'saga', 'penalty': 'l2', 'max_iter': 500, 'class_weight': 'balanced', 'C': 10}
Logistic Regression Best CV Score:  0.8483988886715265


In [30]:
best_svm = svm_search.best_estimator_
best_lr = lr_search.best_estimator_
best_lr

,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add an L2 penalty term and it is the default choice;- `'l1'`: add an L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` and `C` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'`, `l1_ratio` set to any float between 0 and 1 for `penalty='elasticnet'`, and `C=np.inf` for `penalty=None`.",'l2'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",10
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",'balanced'
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following aspects:- 'lbfgs' is a good default solver because it works reasonably well for a wide class of problems.- For :term:`multiclass` problems (`n_classes >= 3`), all solvers except 'liblinear' minimize the full multinomial loss, 'liblinear' will raise an error.- 'newton-cholesky' is a good choice for `n_samples` >> `n_features * n_classes`, especially with one-hot encoded categorical features with rare categories. Be aware that the memory usage of this solver has a quadratic dependency on `n_features * n_classes` because it explicitly computes the full Hessian matrix.- For small datasets, 'liblinear' is a good choice, whereas 'sag' and 'saga' are faster for large ones;- 'liblinear' can only handle binary classification by default. To apply a one-versus-rest scheme for the multiclass setting one can wrap it with the :class:`~sklearn.multiclass.OneVsRestClassifier`... warning:: The choice of the algorithm depends on the penalty chosen (`l1_ratio=0` for L2-penalty, `l1_ratio=1` for L1-penalty and `0 < l1_ratio < 1` for Elastic-Net) and on (multinomial) multiclass support: ================= ======================== ====================== solver l1_ratio multinomial multiclass ================= ======================== ====================== 'lbfgs' l1_ratio=0 yes 'liblinear' l1_ratio=1 or l1_ratio=0 no 'newton-cg' l1_ratio=0 yes 'newton-cholesky' l1_ratio=0 yes 'sag' l1_ratio=0 yes 'saga' 0<=l1_ratio<=1 yes ================= ======================== ======================.. note:: 'sag' and 'saga' fast convergence is only guaranteed on features with approximately the same scale. You can preprocess the data with a scaler from :mod:`sklearn.preprocessing`... seealso:: Refer to the :ref:`User Guide <Logistic_regression>` for more information regarding :class:`LogisticRegression` and more specifically the :ref:`Table <logistic_regression_solvers>` summarizing solver/penalty supports... versionadded:: 0.17 Stochastic Average Gradient (SAG) descent solver. Multinomial support in version 0.18... versionadded

In [31]:
svm_pred = best_svm.predict(X_test_tfidf)
lr_pred = best_lr.predict(X_test_bow)

In [32]:
print("========== Logistic Regression ==========\n")

print("Accuracy :", accuracy_score(y_test, lr_pred))
print("Precision:", precision_score(y_test, lr_pred,average='macro'))
print("Recall   :", recall_score(y_test, lr_pred,average='macro'))
print("F1 Score :", f1_score(y_test, lr_pred,average='macro'))

print("\nClassification Report\n")
print(classification_report(y_test, lr_pred))

print("\nConfusion Matrix\n")
print(confusion_matrix(y_test, lr_pred))

========== Logistic Regression ==========

Accuracy : 0.88375
Precision: 0.8428106594011638
Recall   : 0.8598220267279083
F1 Score : 0.8506904540661612

Classification Report

              precision    recall  f1-score   support

           0       0.94      0.91      0.92       946
           1       0.86      0.87      0.87       427
           2       0.79      0.83      0.81       296
           3       0.71      0.81      0.76       113
           4       0.84      0.83      0.84       397
           5       0.91      0.91      0.91      1021

    accuracy                           0.88      3200
   macro avg       0.84      0.86      0.85      3200
weighted avg       0.89      0.88      0.88      3200


Confusion Matrix



[[857  31   8   5  19  26]
 [ 19 373   4   2  16  13]
 [  3   2 246   1   4  40]
 [  3   0   0  91  16   3]
 [ 14  17   2  22 330  12]
 [ 17  10  50   7   6 931]]


In [33]:
print("========== SVM ==========\n")

print("Accuracy :", accuracy_score(y_test, svm_pred))
print("Precision:", precision_score(y_test, svm_pred,average='macro'))
print("Recall   :", recall_score(y_test, svm_pred,average='macro'))
print("F1 Score :", f1_score(y_test, svm_pred,average='macro'))

print("\nClassification Report\n")
print(classification_report(y_test, svm_pred))

print("\nConfusion Matrix\n")
print(confusion_matrix(y_test, svm_pred))

========== SVM ==========

Accuracy : 0.8965625
Precision: 0.8578481197993337
Recall   : 0.8875034114815818
F1 Score : 0.8709444140505206

Classification Report

              precision    recall  f1-score   support

           0       0.95      0.91      0.93       946
           1       0.87      0.92      0.89       427
           2       0.75      0.91      0.82       296
           3       0.78      0.83      0.80       113
           4       0.86      0.87      0.86       397
           5       0.95      0.89      0.92      1021

    accuracy                           0.90      3200
   macro avg       0.86      0.89      0.87      3200
weighted avg       0.90      0.90      0.90      3200


Confusion Matrix

[[862  31  10   4  21  18]
 [ 12 391   5   1  12   6]
 [  3   3 268   1   4  17]
 [  3   0   1  94  14   1]
 [ 10  17   2  14 346   8]
 [ 19   9  72   7   6 908]]


In [34]:
comparison = pd.DataFrame({
    "Model": ["Logistic Regression", "SVM"],
    "Accuracy": [
        accuracy_score(y_test, lr_pred),
        accuracy_score(y_test, svm_pred)
    ],
    "Precision": [
        precision_score(y_test, lr_pred,average='macro'),
        precision_score(y_test, svm_pred,average='macro')
    ],
    "Recall": [
        recall_score(y_test, lr_pred,average='macro'),
        recall_score(y_test, svm_pred,average='macro')
    ],
    "F1 Score": [
        f1_score(y_test, lr_pred,average='macro'),
        f1_score(y_test, svm_pred,average='macro')
    ]
})

comparison.sort_values(by="F1 Score", ascending=False)

,Model,Accuracy,Precision,Recall,F1 Score
1,SVM,0.896563,0.857848,0.887503,0.870944
0,Logistic Regression,0.883750,0.842811,0.859822,0.850690


In [35]:
print(confusion_matrix(y_test, lr_pred))
print(classification_report(y_test, lr_pred))

[[857  31   8   5  19  26]
 [ 19 373   4   2  16  13]
 [  3   2 246   1   4  40]
 [  3   0   0  91  16   3]
 [ 14  17   2  22 330  12]
 [ 17  10  50   7   6 931]]
              precision    recall  f1-score   support

           0       0.94      0.91      0.92       946
           1       0.86      0.87      0.87       427
           2       0.79      0.83      0.81       296
           3       0.71      0.81      0.76       113
           4       0.84      0.83      0.84       397
           5       0.91      0.91      0.91      1021

    accuracy                           0.88      3200
   macro avg       0.84      0.86      0.85      3200
weighted avg       0.89      0.88      0.88      3200



In [36]:
import joblib

joblib.dump(best_svm, "emotion_svm.pkl")
joblib.dump(tfidf_vectorizer, "tfidf_vectorizer.pkl")

['tfidf_vectorizer.pkl']